# Evaluation Dataset & Benchmark Notebook
## ## C²DH Oral History Workflow — Multilingual Goldstandard Evaluation (v1.2.0-draft)

**Author:** Klaus Behnam Shad
**Affiliation:** Centre for Contemporary and Digital History (C²DH), Université du Luxembourg
**Project context:** DINOH — Digital Infrastructure for Oral History (2024–2027)
**Date:** June 2026
**Status:** v1.2.0-draft for internal review
**Corpus/data version:** 1.1 (unchanged — versioned separately from the repository release)

---

### Purpose

This notebook documents and operationalises the construction of a **multilingual goldstandard evaluation dataset** for the C²DH Oral History Workflow. It serves three connected purposes:

1. **Construction** — assembling a small but methodologically defensible corpus of *synthetic* interview excerpts in seven target languages (DE, EN, FR, IT, ES, LB, PT), each annotated against the Luxembourg Minimal Metadata Schema (LuxOH-CMDI) and an analytical layer compatible with the existing `qda-mini-workflow` data model and the Jupyter analysis pipeline (L1/L2 segmentation).
2. **Evaluation** — running selected open-weight language models (Mistral, OLMo 2, Gemma 3) against the goldstandard for two tasks: minimal metadata extraction and L1/L2 thematic segmentation.
3. **Documentation** — producing reproducible, citable evaluation metrics suitable for methodological publication and internal infrastructure decisions.

### Methodological frame

This notebook implements a **researcher-in-the-loop** evaluation logic: AI outputs are evaluated *against* human-annotated ground truth, never used to generate or modify it. The goldstandard remains a human artefact. KI-use is bounded to suggestion and segmentation; final interpretation remains with the researcher, consistent with the C²DH ethical framework and the project's stated boundaries.

### Synthetic data declaration

All interview excerpts in this dataset are **synthetic**. They were generated for the sole purpose of methodological evaluation. They do not represent real persons, real testimonies, or real events. Every excerpt carries a `synthetic: true` flag at the record level. No real interview material — neither from DINOH nor from any other C²DH project — has been used in the construction of this evaluation set. Should the dataset be published or shared, this declaration must travel with it.

### Scope boundary

This notebook concerns the **C²DH institutional oral history workflow** exclusively. Other research or development contexts the author may be involved in are out of scope and are not referenced.


## Table of contents

0. Setup & dependencies
1. Schemas — Metadata and evaluation contract
2. Goldstandard data model — schema-compatible extension for analytical annotations
3. The synthetic corpus (seven languages, 28 excerpts)
4. Validation of the goldstandard
5. Model abstraction layer — pluggable backends
6. Task 1 — Metadata extraction evaluation
7. Task 2 — L1 thematic segmentation evaluation (WindowDiff, Pk)
8. Task 3 (stub) — L2 interpretive layer agreement (qualitative)
9. Reporting — CSV and LaTeX export
10. Extension points — how to grow this notebook

---


## 0. Setup & dependencies

Open-source dependencies only. The notebook is designed to run locally for goldstandard validation, and on Lux-HPC for the model inference cells (Section 5–7). The model inference cells are isolated so that the goldstandard can be reviewed independently of any compute environment.


In [ ]:
# Dependencies are declared in environment.yml (conda) and requirements.txt (pip).
# Recommended workflow: create the pinned environment, register the kernel, then
# launch Jupyter. See README.md > Quick start.
#
# If you are in an ad-hoc kernel and only miss the one pip-only dependency,
# uncomment the two lines below:
# import sys
# !{sys.executable} -m pip install segeval==2.0.11


In [ ]:
# ── Imports ──────────────────────────
import sys
import json
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

import pandas as pd
import jsonschema

# ── Project paths (robust whether cwd is the repo root or notebooks/) ──
def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "corpus_synthetic.json").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
DATA_DIR     = PROJECT_ROOT / "data"
REPORTS_DIR  = PROJECT_ROOT / "reports"
SCHEMA_DIR   = PROJECT_ROOT / "schemas"

# Make the in-repo package importable without an editable install.
_SRC = str(PROJECT_ROOT / "src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

# Core evaluation logic lives in the tested package (src/oh_eval), not in this
# notebook, so the notebook and the test suite cannot drift apart.
from oh_eval.metrics import (
    SEGEVAL_AVAILABLE,
    CONTROLLED_FIELDS, EXACT_MATCH_FIELDS, LIST_FIELDS, FREE_TEXT_FIELDS,
    evaluate_metadata, evaluate_segmentation,
)
from oh_eval.schema import (
    load_schema, validate_luxoh_record, validate_eval_record,
)
from oh_eval.corpus import (
    load_corpus, records, language_counts, validate_corpus_shape,
)
from oh_eval.reports import (
    write_validation_report, write_task1_reports, write_task2_reports,
    write_segmentation_latex,
)

print(f"Project root:      {PROJECT_ROOT}")
print(f"segeval available: {SEGEVAL_AVAILABLE}")


## 1. Schemas — institutional minimum and evaluation contract

Validation in this notebook uses a single source of truth: the JSON Schema
files in `schemas/`, loaded through `oh_eval.schema`. Nothing is re-defined
inline, so the notebook cannot drift from the canonical schemas or the test
suite.

Two layered contracts apply:

- **Metadata** — the institutional minimum: the published LuxOH-CMDI schema,
  maintained in the project's GitLab repository and archived on Zenodo
  (`luxoh_minimal_metadata.schema.json`).
- **Evaluation record contract** — the research/benchmark superset
  (`oh_eval_record.schema.json`): additionally requires `synthetic: true`, a
  non-empty `transcript`, the `interpretive_layer`, and per-segment
  `analytical_annotations`, with strict `HH:MM:SS` timecodes. Segment
  contiguity is checked in Python (`oh_eval.schema.contiguity_errors`).

Both are JSON Schema draft-07.

In [ ]:
# Schemas are loaded from schemas/ via oh_eval.schema (single source of truth).
# This cell only *displays* them; it does not define any validator.
luxoh_schema = load_schema("luxoh")   # Metadata (institutional minimum)
eval_schema  = load_schema("eval")    # evaluation / benchmark contract

print("Metadata — required fields:")
print("  ", luxoh_schema["required"])
print("\nEvaluation contract — required fields:")
print("  ", eval_schema["required"])
print("\nEval contract adds (vs. Metadata):",
      sorted(set(eval_schema["required"]) - set(luxoh_schema["required"])))


## 2. Goldstandard data model — schema-compatible extension

The Metadata schema (LuxOH-CMDI) defines the **institutional minimum**. For evaluation purposes we extend each record with:

- a top-level `synthetic` flag (mandatory for this dataset),
- a top-level `transcript` field (the excerpt text — not part of the Metadata schema, present here only for the evaluation context),
- a top-level `interpretive_layer` field (L2 interpretive annotation, separate from L1 in `timecoded_segments`),
- per-segment `analytical_annotations` inside each `timecoded_segments` entry, mapping to the `selected_*` fields of `qda-mini-workflow`.

Since the Metadata schema's JSON Schema does not set `additionalProperties: false`, these extensions are **schema-compatible**: a goldstandard record remains valid LuxOH-CMDI. The extensions sit *on top of* the institutional minimum without replacing it.

### `analytical_annotations` structure (per L1 segment)

Mapping to `qda-mini-workflow` `selected_*` fields:

```python
{
  "selected_topic":              "<str>",          # e.g. "border crossing"
  "selected_frame":              "<str>",          # e.g. "moral economy"
  "selected_rhetorical_pattern": "<str>",          # e.g. "contrast"
  "selected_discursive_position":"<str>",          # e.g. "marginalised voice"
  "gold_rationale":              "<str>",          # researcher justification
  "annotator_id":                "<str>",          # for IAA later
  "alternative_codings":         ["<str>", ...]    # optional, for ambiguity
}
```

### `interpretive_layer` structure (L2, per record)

```python
{
  "narrative_arc":          "<str>",   # one-sentence trajectory
  "central_tension":        "<str>",   # what is at stake
  "positioning_summary":    "<str>",   # Bamberg-informed
  "gold_rationale":         "<str>",
  "annotator_id":           "<str>"
}
```

**L3 (Oberthema) is deliberately excluded from the goldstandard.** It is reserved for the researcher's hermeneutic act and AI use is excluded here as a matter of methodological policy.


In [ ]:
# Dataclasses for goldstandard records — purely for readability;
# the canonical format is the validated JSON.

@dataclass
class AnalyticalAnnotation:
    selected_topic: str
    selected_frame: str
    selected_rhetorical_pattern: str
    selected_discursive_position: str
    gold_rationale: str
    annotator_id: str = "[ANNOTATOR_1]"
    alternative_codings: list[str] = field(default_factory=list)

@dataclass
class L1Segment:
    start: str
    end: str
    label: str
    analytical_annotations: Optional[AnalyticalAnnotation] = None

@dataclass
class InterpretiveLayer:
    narrative_arc: str
    central_tension: str
    positioning_summary: str
    gold_rationale: str
    annotator_id: str = "[ANNOTATOR_1]"

@dataclass
class GoldRecord:
    # Metadata required
    record_id: str
    interview_date: str
    interviewer: str
    consent_status: str
    accessRights: str
    title: str
    language: str
    # Metadata optional
    interviewee_display: str = ""
    spatial: str = ""
    keywords: list[str] = field(default_factory=list)
    abstract: str = ""
    timecoded_segments: list[L1Segment] = field(default_factory=list)
    # Evaluation extensions
    synthetic: bool = True
    transcript: str = ""
    interpretive_layer: Optional[InterpretiveLayer] = None

    def to_dict(self) -> dict:
        d = asdict(self)
        # Drop None entries
        if d.get("interpretive_layer") is None:
            d.pop("interpretive_layer", None)
        for seg in d.get("timecoded_segments", []):
            if seg.get("analytical_annotations") is None:
                seg.pop("analytical_annotations", None)
        return d


## 3. The synthetic corpus

28 synthetic excerpts, 4 per language (DE, EN, FR, IT, ES, LB, PT).
Each excerpt is ~200–400 words. Each excerpt is annotated against:
- Metadata (record-level)
- L1 segments (3–6 per excerpt) with `analytical_annotations`
- L2 `interpretive_layer` (one per excerpt)

### Thematic axes

Per language, the four excerpts cover two thematic axes × two register variations:

| Slot | Theme         | Register             |
|------|---------------|----------------------|
| 1    | Border / mobility | colloquial      |
| 2    | Border / mobility | reflective      |
| 3    | Labour biography  | colloquial      |
| 4    | Labour biography  | reflective      |

This variation prevents trivial pattern-matching by keyword-based topic rules and surfaces register effects on segmentation.

### Annotator note

For this v1 draft, all annotations are by a single annotator ([ANNOTATOR_1], annotator_id `[ANNOTATOR_1]`). A second annotator pass is recommended before publication-grade use; the data model supports this without restructuring (see Section 10).


### Low-resource language tracking

Luxembourgish (`ltz`) is treated as a **low-resource language** with separate evaluation tracking. Portuguese (`por`) was added in v1.1 to reflect the demographic reality of Luxembourg — speakers of Portuguese descent constitute a major segment of the population, and any evaluation of OH infrastructure that omits Portuguese would be ethnographically incomplete. Portuguese is not low-resource in the same technical sense as Luxembourgish (it is well-supported in current multilingual models), but it is methodologically essential for the Luxembourg context. See Section 6/7 for sprachspezifische Auswertung.

---

The corpus is loaded from `corpus_synthetic.json` (generated below for self-containment).


In [ ]:
# Corpus loading + shape checks live in oh_eval.corpus (see Section 0).
CORPUS_PATH = DATA_DIR / "corpus_synthetic.json"
corpus_data = load_corpus(CORPUS_PATH)

shape_errors = validate_corpus_shape(corpus_data)
if shape_errors:
    raise ValueError("Corpus shape problems: " + "; ".join(shape_errors))

corpus = records(corpus_data)
print(f"Loaded {len(corpus)} synthetic records from {CORPUS_PATH}")
print("Per-language counts:", language_counts(corpus))


### Hinweis zur Record-ID-Konvention

Das Metadata-Schema (LuxOH-CMDI) definiert das Pattern `^[0-9]{4}-[0-9]{2}-[0-9]{2}_[A-Z]{2,10}_[0-9]{4}$` für `record_id` und erlaubt im mittleren Akronym-Slot ausschließlich Großbuchstaben (`A–Z`). Da das institutionelle Akronym **C²DH** in seiner ASCII-Form `C2DH` eine Ziffer enthält, würde es das Pattern nicht erfüllen.

In dieser v1-Goldstandard-Sammlung wird daher pragmatisch `CDH` als Repository-Präfix verwendet. Vor produktivem Einsatz sollte abgestimmt werden, ob das Schema-Pattern auf `[A-Z0-9]{2,10}` erweitert wird (damit `C2DH` zulässig wäre) oder ob `CDH` als institutioneller Kürzel bestehen bleibt. Dies ist eine **infrastrukturelle Entscheidung**, keine methodische, und wird hier ausschließlich dokumentiert.

---


## 4. Validation of the goldstandard

Every record is validated against **both** contracts, both loaded from
`oh_eval.schema`:

- the **Metadata** (institutional minimum), and
- the **evaluation record contract** (the research/benchmark superset: the
  `synthetic` flag, transcript, interpretive layer, per-segment analytical
  annotations, strict timecodes, and — in Python — segment contiguity).

Both are blockers: errors must be resolved before the dataset is shared. Two
reports are written in Section 9 (`luxoh_validation_report.csv` and
`oh_eval_validation_report.csv`).

In [ ]:
def _validation_rows(validate):
    rows = []
    for record in corpus:
        errors = validate(record)
        rows.append({
            "record_id": record["record_id"],
            "language": record["language"],
            "valid": len(errors) == 0,
            "n_errors": len(errors),
            "errors": "; ".join(errors) if errors else "",
        })
    return pd.DataFrame(rows)

# Metadata (institutional minimum)
vdf = _validation_rows(validate_luxoh_record)
# Evaluation record contract (structural schema + segment contiguity)
edf = _validation_rows(validate_eval_record)

for name, df in [("Metadata", vdf), ("Evaluation contract", edf)]:
    print(f"{name}: {len(df)} records — {int(df['valid'].sum())} valid, "
          f"{int((~df['valid']).sum())} invalid")

display(vdf)
display(edf)


## 5. Model abstraction layer

A thin abstraction over model backends. The evaluation logic does not depend on *how* a model is served (local HuggingFace, vLLM on Lux-HPC, Ollama, etc.). To add a new backend, register a new entry in `MODEL_REGISTRY` and implement `run_model(model_name, prompt, segment_text)`.

For the v1 draft, the registry contains placeholder entries for **Mistral open**, **OLMo 2**, and **Gemma 3**. The actual inference calls are stubbed — they return a deterministic placeholder result so that the downstream evaluation logic can be tested end-to-end without an HPC connection. The HPC backend will be wired in when Lux-HPC access is provisioned.

### Why open weights only

Per project policy: reproducibility, no data egress to third parties, GDPR-compatible processing on institutional infrastructure.


In [ ]:
MODEL_REGISTRY = {
    "mistral-7b-instruct-v0.3": {
        "family": "mistral",
        "weights_url": "https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3",
        "license": "Apache-2.0",
        "backend": "stub"  # change to "vllm" / "hf" / "ollama" on HPC
    },
    "olmo-2-7b-instruct": {
        "family": "olmo",
        "weights_url": "https://huggingface.co/allenai/OLMo-2-1124-7B-Instruct",
        "license": "Apache-2.0",
        "backend": "stub"
    },
    "gemma-3-12b-it": {
        "family": "gemma",
        "weights_url": "https://huggingface.co/google/gemma-3-12b-it",
        "license": "Gemma Terms of Use",
        "backend": "stub"
    }
}

def run_model(model_name: str, prompt: str, payload: dict) -> dict:
    """
    Run a model on a given prompt and payload.
    Returns the raw model output as a dict (parsed JSON if applicable).

    For backend='stub': returns a deterministic placeholder for pipeline testing.
    For backend='vllm'/'hf': implement actual inference (TODO on HPC).
    """
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model: {model_name}")
    backend = MODEL_REGISTRY[model_name]["backend"]

    if backend == "stub":
        # Returns a plausible-shaped stub so downstream evaluation runs.
        # Replace with real inference once HPC is available.
        return _stub_output(model_name, prompt, payload)
    else:
        raise NotImplementedError(
            f"Backend '{backend}' not yet implemented. "
            "Wire in vLLM/HF/Ollama once Lux-HPC access is available."
        )


def _stub_output(model_name: str, prompt: str, payload: dict) -> dict:
    """Deterministic placeholder: returns the gold answer perturbed slightly.

    Purpose: smoke-test the evaluation pipeline end-to-end without a real model.
    The stub deliberately introduces small errors so that metrics are non-trivial.
    """
    task = payload.get("_task")
    if task == "metadata":
        # Return gold metadata but drop the abstract and shorten keywords (stub error)
        record = payload["_gold_record"]
        return {
            "record_id": record["record_id"],
            "interview_date": record["interview_date"],
            "interviewer": record["interviewer"],
            "consent_status": record["consent_status"],
            "accessRights": record["accessRights"],
            "title": record["title"],
            "language": record["language"],
            "spatial": record.get("spatial", ""),
            "keywords": record.get("keywords", [])[:2],  # error: truncates
            "abstract": "",                              # error: omits
        }
    elif task == "segmentation":
        # Introduce ONE real boundary error: merge the final two gold segments
        # (a "missed boundary"). Unlike dropping the last segment, this actually
        # removes an interior boundary, so WindowDiff/Pk are non-trivial under the
        # offset-based masses.
        record = payload["_gold_record"]
        segments = record.get("timecoded_segments", [])
        if len(segments) >= 3:
            merged = [
                {"start": s["start"], "end": s["end"], "label": s["label"]}
                for s in segments[:-2]
            ] + [{
                "start": segments[-2]["start"],
                "end": segments[-1]["end"],
                "label": segments[-2]["label"],
            }]
            return {"segments": merged}
        return {"segments": segments}
    else:
        return {"_stub": True, "note": "no task-specific stub defined"}


## 6. Task 1 — Metadata extraction evaluation

The task: given an interview transcript, extract Metadata fields.

### Metrics

Per field, exact-match precision/recall/F1 for controlled fields, and a manual qualitative tag (`match` / `partial` / `mismatch`) for free-text fields (`abstract`, `title`). Free-text fields are *not* auto-scored with embedding similarity in v1 — that would be methodologically circular if used for goldstandard validation. Embedding scoring can be added as a *supplementary* signal in a later version.

### Per-language evaluation, no pooled scores

Following the project's methodological position on Luxembourgish as a low-resource language, metrics are reported **per language**. A pooled cross-language F1 is not reported.


In [ ]:
# Field constants and evaluate_metadata() are imported from
# oh_eval.metrics (Section 0). The loop below is the run harness.

# Run task 1 across all records, all models
task1_rows = []
for model_name in MODEL_REGISTRY:
    for record in corpus:
        payload = {"_task": "metadata", "_gold_record": record}
        prediction = run_model(model_name, prompt="<<metadata extraction prompt placeholder>>", payload=payload)
        outcomes = evaluate_metadata(record, prediction)
        for field_, o in outcomes.items():
            row = {
                "model": model_name,
                "language": record["language"],
                "record_id": record["record_id"],
                "field": field_
            }
            row.update(o)
            task1_rows.append(row)

task1_df = pd.DataFrame(task1_rows)
print(f"Task 1: {len(task1_df)} field-level evaluations across {len(MODEL_REGISTRY)} models and {len(corpus)} records")
display(task1_df.head(12))


In [ ]:
# Aggregate: exact-match accuracy per (model, language, field)
exact_fields = [f for f in EXACT_MATCH_FIELDS]
agg_rows = []
for (model, language, field_), grp in task1_df[task1_df["field"].isin(exact_fields)].groupby(["model", "language", "field"]):
    accuracy = grp["match"].mean()
    n = len(grp)
    agg_rows.append({"model": model, "language": language, "field": field_, "accuracy": accuracy, "n": n})
exact_agg = pd.DataFrame(agg_rows)

# Pivot for readability
pivot = exact_agg.pivot_table(
    index=["model", "language"], columns="field", values="accuracy"
).round(2)
print("Exact-match accuracy by (model, language, field) — Task 1")
display(pivot)


## 7. Task 2 — L1 thematic segmentation evaluation

The task: given an interview transcript, propose L1 segment boundaries.

### Metrics

- **WindowDiff** and **Pk** (Beeferman et al. 1999; Pevzner & Hearst 2002), computed via `segeval`. These metrics tolerate near-misses on boundary placement and are the standard for segmentation evaluation.
- **Boundary count agreement** (|gold segments| vs |predicted segments|) as a coarse sanity check.

### Notes

WindowDiff and Pk both treat segmentation as a sequence of *masses* (segment lengths) over a shared unit axis. The corpus provides **timecoded** (not text-aligned) boundaries, so each segment's end timecode is projected onto the transcript's character axis proportionally to its position in the interview timeline. Gold and prediction are mapped with the *same* timeline, so both produce masses summing to the transcript length and the metrics reflect **boundary placement** rather than the segment-count artefact of an even split. Character units are more robust across the seven languages than whitespace tokenisation, particularly for Luxembourgish where tokenisers may be inconsistent.


In [ ]:
# segments_to_masses() / evaluate_segmentation() are imported from
# oh_eval.metrics (Section 0). The loop below is the run harness.

task2_rows = []
for model_name in MODEL_REGISTRY:
    for record in corpus:
        payload = {"_task": "segmentation", "_gold_record": record}
        prediction = run_model(model_name, prompt="<<segmentation prompt placeholder>>", payload=payload)
        outcome = evaluate_segmentation(record, prediction)
        outcome["model"] = model_name
        outcome["language"] = record["language"]
        outcome["record_id"] = record["record_id"]
        task2_rows.append(outcome)

task2_df = pd.DataFrame(task2_rows)
print("Task 2: L1 segmentation results")
display(task2_df.head(12))


In [ ]:
# Per-language aggregation for Task 2
seg_agg = task2_df.groupby(["model", "language"]).agg(
    mean_window_diff=("window_diff", "mean"),
    mean_pk=("pk", "mean"),
    mean_count_diff=("count_diff", "mean"),
    n=("record_id", "count")
).round(3)
print("Mean segmentation metrics by (model, language) — Task 2")
display(seg_agg)


## 8. Task 3 (stub) — L2 interpretive layer agreement

L2 interpretive output cannot be evaluated by automatic metrics alone. The intended evaluation is a **qualitative 3-level agreement rating** (`agreement` / `partial` / `disagreement`) by the researcher, with documented rationale.

For v1 we provide the data structure and a manual review template. Filling it is researcher work, not model work — by design. The stub below shows the intended workflow.


In [ ]:
def l2_review_template(gold_record: dict, prediction: dict) -> dict:
    """Return a review template — to be filled in by the researcher."""
    return {
        "record_id": gold_record["record_id"],
        "language": gold_record["language"],
        "gold_l2": gold_record.get("interpretive_layer"),
        "predicted_l2": prediction.get("interpretive_layer", "<<not yet produced>>"),
        "agreement_narrative_arc":       "TODO: agreement | partial | disagreement",
        "agreement_central_tension":     "TODO: agreement | partial | disagreement",
        "agreement_positioning_summary": "TODO: agreement | partial | disagreement",
        "researcher_rationale": "TODO: free-text rationale (mandatory)",
        "annotator_id": "[ANNOTATOR_1]"
    }

# Demo: produce review templates for the first record across models (stubbed)
demo_templates = []
for model_name in MODEL_REGISTRY:
    demo_templates.append(l2_review_template(corpus[0], {}))
print(f"Produced {len(demo_templates)} L2 review templates (stub).")


## 9. Reporting — CSV and LaTeX export

Two outputs:
1. **CSV** — full row-level results for internal review.
2. **LaTeX** — publication-ready summary tables.

Outputs land in `./reports/` (created if absent).


In [ ]:
# Report writers live in oh_eval.reports (see Section 0).
reports_dir = REPORTS_DIR
reports_dir.mkdir(exist_ok=True)

write_task1_reports(task1_df, exact_agg, reports_dir)
write_task2_reports(task2_df, seg_agg, reports_dir)
write_validation_report(vdf, reports_dir / "luxoh_validation_report.csv")
write_validation_report(edf, reports_dir / "oh_eval_validation_report.csv")
write_segmentation_latex(seg_agg, reports_dir / "task2_segmentation_summary.tex")

print("Wrote:")
for p in sorted(reports_dir.iterdir()):
    print("  -", p)


## 10. Extension points — how to grow this notebook

The notebook is intentionally modular. The following extensions can be added without restructuring:

### 10.1 More languages / more excerpts (horizontal scaling)

Append new records to `corpus_synthetic.json`. The validation, evaluation, and reporting cells iterate over `corpus` directly; no other change required. For an additional language (e.g. Portuguese for a Greater Region variant), add the ISO 639-3 code (`por`) in records and update per-language reporting filters if filtering is desired.

### 10.2 New evaluation tasks (vertical scaling)

Each evaluation task is a `evaluate_<task>(gold, prediction) -> dict` function. To add a task — for example, affect coding, discursive position prediction, rhetorical-pattern classification — implement a new function with the same signature, and loop over the corpus following the Task 1 / Task 2 template. Suggested next tasks in order of methodological tractability:

1. **Topic classification per segment** (already in goldstandard as `selected_topic`) — micro-F1 against `qda-mini-workflow` topic vocabulary.
2. **Rhetorical-pattern classification** — likewise, against the controlled rhetorical_pattern vocabulary.
3. **Discursive position prediction** — more interpretive; report alongside qualitative agreement.

### 10.3 Real model backends

Replace `_stub_output` with real inference. The cleanest path on Lux-HPC is vLLM-served local models with an HTTP client in `run_model`. The function signature does not change. For local pre-HPC testing, swap to `transformers` with `pipeline("text-generation", ...)`.

### 10.4 Second annotator and inter-annotator agreement

The data model already includes `annotator_id`. To add a second annotator pass:

1. Duplicate `corpus_synthetic.json` as `corpus_synthetic_annotator2.json` and re-annotate (the annotator must not see the first pass).
2. Implement `compute_iaa(annot1_corpus, annot2_corpus)` using Cohen's κ (per controlled field) and Krippendorff's α (overall). Suggested library: `nltk.metrics.agreement` or `krippendorff`.
3. Report IAA per language and per field. Below ~0.6 on Cohen's κ is a signal to revise the coding manual.

### 10.5 Active goldstandard expansion

For larger goldstandards (e.g. 100+ excerpts), the synthetic-generation approach reaches its useful limit. A real-data path would require (a) IRB/DPO approval at C²DH, (b) explicit informed consent covering AI evaluation use, (c) pseudonymisation following the LuxOH-CMDI workflow. The evaluation infrastructure does not change — only the data source.

### 10.6 Reporting and publication

The CSV outputs in `./reports/` are designed to be the data source for figures and tables in a methods paper. The LaTeX table generated in Section 9 can be included directly in a manuscript. For a publication-grade run, all stub outputs in Sections 6–7 must be replaced with real model inferences.

### 10.7 What this notebook deliberately does *not* do

- **No AI-generated goldstandard.** Goldstandards are human artefacts. A future version could introduce model-assisted *pre-annotation* with mandatory human review, structurally analogous to the `suggested_*` / `selected_*` distinction in `qda-mini-workflow`. The boundary between AI suggestion and human selection must remain visible in the data.
- **No L3 evaluation.** L3 (Oberthema) is reserved for the hermeneutic act of the researcher and is excluded from AI evaluation by methodological choice.
- **No cross-language pooled scores.** Reporting per language is a deliberate methodological position, not an oversight.

---

### Versioning

Current notebook version: see `NOTEBOOK_VERSION` in Section 0. Changes should be documented in a `CHANGELOG.md` alongside the notebook in the institutional repository.

---

**End of notebook.**
